In [0]:
from pyspark.sql.functions import lit, current_date, col, date_sub, row_number, upper, sum, monotonically_increasing_id
from pyspark.sql.window import Window

def data_quality_checks(start_date,end_date):
    try:
        customers=spark.read.table("brazilian_ecommerce.bronze.customers").take(1)
        sellers=spark.read.table("brazilian_ecommerce.bronze.sellers").take(1)
        orders=spark.sql(f"select * from Brazilian_Ecommerce.bronze.orders where to_date(order_purchase_timestamp,'yyy-MM-dd') between '{start_date}' and '{end_date}'").take(1)
        products=spark.read.table("brazilian_ecommerce.bronze.products").take(1)
        product_category=spark.read.table("brazilian_ecommerce.bronze.product_category").take(1)
        df = spark.read.table("brazilian_ecommerce.bronze.orders")

        # Check if the bronze tables are empty
        if len(customers)==0 or len(sellers)==0 or len(products)==0 or len(orders)==0 or len(product_category)==0:
            raise Exception("Data quality check failed: No new data to process")
        # Check if the bronze tables have negative values for KPIs
        elif df.filter((col("price") < 0) | (col("freight_value") < 0) | (col("payment_installments") < 0) | (col("payment_value") < 0)).take(1):
            raise Exception("Data quality check failed: negative values for KPIs detected" )
        else:
            print("Data quality checks passed")
    except Exception as e:
        raise Exception(f"Error: {e}")


def silver_ingestion(start_date,end_date):
     
    try:
        # Step 1 - Perform data quality checks
       
        data_quality_checks(start_date,end_date)

        # # Step 2 - Create orders silver table
        bronze_orders = spark.table("brazilian_ecommerce.bronze.orders")
        window = Window.partitionBy("order_id").orderBy(col("order_purchase_timestamp").desc())

        silver_orders = (
            bronze_orders
            .filter(
                (col("order_purchase_timestamp") > lit(start_date)) &
                (col("order_purchase_timestamp") < lit(end_date)) &
                col("order_id").isNotNull() &
                col("customer_id").isNotNull() &
                col("seller_id").isNotNull() &
                col("product_id").isNotNull() &
                col("price").isNotNull() &
                col("freight_value").isNotNull() &
                col("payment_value").isNotNull() &
                col("payment_installments").isNotNull()
            )
            .withColumn("rn", row_number().over(window))
            .filter(col("rn") == 1)
            .select(
                "order_item_id","order_id","customer_id","product_id","seller_id",
                upper(col("order_status")).alias("order_status"),
                "order_purchase_timestamp","order_approved_at",
                "order_delivered_carrier_date","order_delivered_customer_date",
                "order_estimated_delivery_date","price","freight_value",
                "payment_type","payment_installments","payment_value"
            )
        )
        silver_orders.write.mode("append").format("delta").saveAsTable("brazilian_ecommerce.silver.orders")

        # # Step 3 - Customers dimension table (SCD Type 2)
        bronze_customers = spark.table("Brazilian_Ecommerce.bronze.customers")\
            .filter(col("customer_id").isNotNull())\
            .withColumn("rn", row_number().over(Window.partitionBy("customer_id").orderBy("customer_id")))\
            .filter(col("rn") == 1)\
            .select("customer_id", "customer_city", "customer_state")

        valid_from = '2016-01-01'

        silver_customers = spark.table("Brazilian_Ecommerce.silver.customers")


        if silver_customers and len(silver_customers.take(1)) > 0:
            current_silver = silver_customers.filter(col("is_active") == 'TRUE')

            changed = bronze_customers.alias("b").join(
                current_silver.alias("s"),
                on="customer_id",
                how="left"
            ).filter(
                (col("s.customer_id").isNull()) |
                (col("b.customer_city") != col("s.customer_city")) |
                (col("b.customer_state") != col("s.customer_state"))
            ).select("b.customer_id","b.customer_city","b.customer_state")
        else:
            changed = bronze_customers

        if changed.count() > 0:
            if silver_customers:
                old_records_to_close = silver_customers.alias("s").join(
                    changed.alias("c"),
                    on="customer_id",
                    how="inner"
                ).select(
                    "s.customer_sk","s.customer_id","s.customer_city","s.customer_state",
                    "s.valid_from",current_date().alias("valid_to"),lit('FALSE').alias("is_active")
                )

                unchanged = silver_customers.alias("s").join(
                    changed.alias("c"),
                    on="customer_id",
                    how="left_anti"
                )

                changed = changed.withColumn("customer_sk", monotonically_increasing_id())\
                                .withColumn("valid_from", lit(valid_from))\
                                .withColumn("valid_to", lit("9999-12-31").cast("date"))\
                                .withColumn("is_active", lit('TRUE'))

                final_dim = unchanged.unionByName(old_records_to_close).unionByName(changed)
            else:
                final_dim = changed.withColumn("customer_sk", monotonically_increasing_id())\
                                .withColumn("valid_from", lit(valid_from))\
                                .withColumn("valid_to", lit("9999-12-31").cast("date"))\
                                .withColumn("is_active", lit('TRUE'))
        else:
            final_dim = silver_customers

        final_dim.write.mode("append").saveAsTable("Brazilian_Ecommerce.silver.customers")

        # Step 4 - Sellers dimension table (SCD Type 2)
        bronze_sellers = spark.table("Brazilian_Ecommerce.bronze.sellers")\
            .filter(col("seller_id").isNotNull())\
            .withColumn("rn", row_number().over(Window.partitionBy("seller_id").orderBy("seller_id")))\
            .filter(col("rn") == 1)\
            .select("seller_id", "seller_state", "seller_city")

        bronze_sellers.createOrReplaceTempView("sellers_bronze_vw")

        # Create staging view with correct SCD2 logic
        spark.sql("""
        CREATE OR REPLACE TEMP VIEW sellers_staging AS


        SELECT 
            src.seller_id,
            src.seller_city,
            src.seller_state,
            src.seller_id AS merge_key
        FROM sellers_bronze_vw src
        JOIN Brazilian_Ecommerce.silver.sellers tgt
            ON src.seller_id = tgt.seller_id
            AND tgt.is_active = 'TRUE'
        WHERE src.seller_city IS DISTINCT FROM tgt.seller_city
        OR src.seller_state IS DISTINCT FROM tgt.seller_state

        UNION ALL


        SELECT 
            src.seller_id,
            src.seller_city,
            src.seller_state,
            NULL AS merge_key
        FROM sellers_bronze_vw src
        LEFT JOIN Brazilian_Ecommerce.silver.sellers tgt
            ON src.seller_id = tgt.seller_id
            AND tgt.is_active = 'TRUE'
        WHERE tgt.seller_id IS NULL
        OR src.seller_city IS DISTINCT FROM tgt.seller_city
        OR src.seller_state IS DISTINCT FROM tgt.seller_state
        """)

        # Merge into target table
        spark.sql(f"""
        MERGE INTO Brazilian_Ecommerce.silver.sellers tgt
        USING sellers_staging src
        ON tgt.seller_id = src.merge_key AND tgt.is_active = 'TRUE'

        WHEN MATCHED THEN 
        UPDATE SET 
            tgt.is_active = 'FALSE',
            tgt.valid_to = current_date()

        WHEN NOT MATCHED THEN 
        INSERT (

            seller_id,
            seller_city,
            seller_state,
            is_active,
            valid_from,
            valid_to
        )
        VALUES (

            src.seller_id,
            src.seller_city,
            src.seller_state,
            'TRUE',
            '{valid_from}',
            DATE '9999-12-31'
        )
        """)

        # Step 5 - Products dimension table (SCD Type 2)
        bronze_products = spark.table("Brazilian_Ecommerce.bronze.products")\
            .filter(col("product_id").isNotNull())\
            .withColumn("rn", row_number().over(Window.partitionBy("product_id").orderBy("product_id")))\
            .filter(col("rn") == 1)\
            .select("product_id", "product_category_name")

        bronze_product_category = spark.read.table("Brazilian_Ecommerce.bronze.product_category")\
            .filter(col("product_category_name").isNotNull())\
            .withColumn("rn", row_number().over(Window.partitionBy("product_category_name").orderBy("product_category_name")))\
            .filter(col("rn") == 1)\
            .select("product_category_name", "product_category_name_english")

        bronze_products.alias("a").join(
            bronze_product_category.alias("b"),
            "product_category_name"
        ).selectExpr(
            "product_id","a.product_category_name","b.product_category_name_english"
        ).createOrReplaceTempView("products_bronze_vw")

        spark.sql("""
       CREATE OR REPLACE TEMP VIEW products_staging AS
        SELECT 

            src.product_id,
            src.product_category_name,
            src.product_category_name_english,
            src.product_id AS merge_key
        FROM products_bronze_vw src
        JOIN Brazilian_Ecommerce.silver.products tgt
            ON src.product_id = tgt.product_id
            AND tgt.is_active = 'TRUE'
        WHERE src.product_category_name IS DISTINCT FROM tgt.product_category_name
        OR src.product_category_name_english IS DISTINCT FROM tgt.product_category_name_english

        UNION ALL

        -- 2. Rows that need INSERT (new OR changed → new version)
        SELECT 
  
            src.product_id,
            src.product_category_name,
            src.product_category_name_english,
            NULL AS merge_key
        FROM products_bronze_vw src
        LEFT JOIN Brazilian_Ecommerce.silver.products tgt
            ON src.product_id = tgt.product_id
            AND tgt.is_active = 'TRUE'
        WHERE tgt.product_id IS NULL
        OR src.product_category_name IS DISTINCT FROM tgt.product_category_name
        OR src.product_category_name_english IS DISTINCT FROM tgt.product_category_name_english
        """)

        spark.sql(f"""
        MERGE INTO Brazilian_Ecommerce.silver.products tgt
        USING products_staging src
        ON tgt.product_id = src.merge_key AND tgt.is_active = 'TRUE'

        WHEN MATCHED THEN 
        UPDATE SET 
            tgt.is_active = 'FALSE', 
            tgt.valid_to = current_date()

        WHEN NOT MATCHED THEN 
        INSERT (
            product_id, 
            product_category_name, 
            product_category_name_english, 
            is_active, 
            valid_from, 
            valid_to
        )
        VALUES (
            src.product_id, 
            src.product_category_name, 
            src.product_category_name_english, 
            'TRUE', 
            '{valid_from}', 
            DATE '9999-12-31'
        )
        """)

        print("All the silver layer tables were created successfully")
    except Exception as e:
        raise Exception(f"Error: {e}")

if __name__ == "__main__":
    print("Silver Ingestion Started")
    start_date='2016-01-01'
    end_date='2018-12-31'
    silver_ingestion(start_date,end_date)
    print("Silver Ingestion Completed")

Silver Ingestion Started
Data quality checks passed
All the silver layer tables were created successfully
Silver Ingestion Completed
